# Домашнее задание № 9

# Задание 1 (10 баллов)

Визуализируйте attention для одного любого текста после нескольких последовательных эпох обучения, чтобы проанализировать как модель учится понимать текст. 
Для этого вам понадобится так изменить код модели из семинара, чтобы Block класс возвращал attention активации (последнее значение wei), а также все остальные классы, которые вызывают Block, чтобы они ожидали, что модель вернет не только out но и wei. В самом верхнеуровневом классе BigramLanguageModel вы можете добавить атрибут last_attentions и в forward перезаписывать его значения последним значением attention (но можно придумать и другой способ). После каждой эпохи вызовите модель на одном примере из датасета и сохраните last_attentions во внешнюю переменную, чтобы потом отдельно заняться визуализацией. Визуализируйте attentions как heatmap'ы (например в searborn). У вас будет attention матрица для каждого слоя и для каждого head в модели. Для каждой нужно будет сделать свой хитмап.

Напишите короткую интерпретацию полученных результатов: какие паттерны в матрицах внимания вы видите? как эти паттерны отличаются между слоями? как они меняются с каждой эпохой? 
Постарайтесь найти какое-то хорошее предложение, для которого у вас заранее есть какое-то представление о связях между словами и которое вы сможете сравнить со скорами внимания (например, какую синтаксическую конструкцию)

Должно получиться что-то похожее на (только несколько для каждой эпохи)
![](https://www.kdnuggets.com/wp-content/uploads/How_to_Visualize_Model_Internals_and_Attention_in_Hugging_Face_Transformers_3.png)

In [2]:
!pip install rusenttokenize
!pip install tokenizers


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from rusenttokenize import ru_sent_tokenize
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer
from tokenizers import decoders

In [5]:
data = pd.read_csv('https://github.com/mannefedov/compling_nlp_hse_course/raw/refs/heads/master/data/lenta_40k.csv.zip')

In [7]:
data.head()

,text,topic
0,Россия должна сотрудничать с Всемирным антидоп...,Спорт
1,Уголовный суд Кувейта 28 июня освободил под за...,Мир
2,Французский журнал Charlie Hebdo опубликовал н...,Интернет и СМИ
3,В Петербурге в доме № 53 по улице Лени Голиков...,Россия
4,"В московском аэропорту ""Домодедово"" задержан г...",Россия


In [9]:
sentences = []
for text in data.text.values:
    sentences.extend(ru_sent_tokenize(str(text)))

with open("corpus.txt", "w", encoding="utf-8") as f:
    for sent in sentences:
        f.write(sent + "\n")

with open("corpus.txt", encoding="utf-8") as f:
    sentences = f.read().splitlines()

print("Количество предложений:", len(sentences))

Количество предложений: 491362


In [10]:
tokenizer = Tokenizer(BPE())
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    special_tokens=["[PAD]", "[BOS]", "[EOS]"],
    end_of_word_suffix='</w>'
)

tokenizer.train(files=["corpus.txt"], trainer=trainer)
tokenizer.save("tokenizer")

tokenizer = Tokenizer.from_file("tokenizer")
tokenizer.decoder = decoders.BPEDecoder()

vocab_size = tokenizer.get_vocab_size()
print("Размер словаря:", vocab_size)

PAD_IDX = tokenizer.token_to_id("[PAD]")

Размер словаря: 30000


In [11]:
def encode(text, tokenizer):
    return [tokenizer.token_to_id("[BOS]")] + \
           tokenizer.encode(text).ids + \
           [tokenizer.token_to_id("[EOS]")]

In [12]:
class Dataset(torch.utils.data.Dataset):
    
    def __init__(self, sentences, tokenizer, max_len=64):
        self.encoded_texts = [
            torch.LongTensor(encode(sent, tokenizer)[-max_len:])
            for sent in sentences
        ]
        
        self.X = torch.nn.utils.rnn.pad_sequence(
            self.encoded_texts,
            padding_value=PAD_IDX,
            batch_first=True
        )
        
        self.length = len(self.encoded_texts)
    
    def __len__(self):
        return self.length
    
    def __getitem__(self, index):
        x = self.X[index][:-1]
        y = self.X[index][1:]
        mask = x != PAD_IDX
        return x, y, mask

In [13]:
n = int(0.9 * len(sentences))

sentences_train = sentences[:n]
sentences_val = sentences[n:]

MAX_LEN = 64

training_set = Dataset(sentences_train, tokenizer, MAX_LEN)
val_set = Dataset(sentences_val, tokenizer, MAX_LEN)

training_generator = torch.utils.data.DataLoader(
    training_set, batch_size=128, shuffle=True
)

val_generator = torch.utils.data.DataLoader(
    val_set, batch_size=128
)

In [14]:
device = 'cuda' if torch.cuda.is_available() else 'cpu' 

block_size = MAX_LEN
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.1
learning_rate = 1e-3

In [15]:
class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer(
            'tril',
            torch.tril(torch.ones(block_size, block_size))
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        B, T, C = x.shape
        
        k = self.key(x)
        q = self.query(x)
        
        wei = q @ k.transpose(-2, -1) * C**-0.5
        
        wei = wei.masked_fill(
            self.tril[:T, :T] == 0,
            float('-inf')
        )
        
        if mask is not None:
            wei = wei.masked_fill(~mask.unsqueeze(1), float('-inf'))
        
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        
        v = self.value(x)
        out = wei @ v
        
        return out

In [16]:
class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList(
            [Head(head_size) for _ in range(num_heads)]
        )
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        out = torch.cat(
            [h(x, mask) for h in self.heads],
            dim=-1
        )
        out = self.proj(out)
        out = self.dropout(out)
        return out

In [17]:
class FeedForward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [18]:
class Block(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, inp):
        x, mask = inp
        
        x = x + self.sa(self.ln1(x), mask)
        x = x + self.ffwd(self.ln2(x))
        
        return x, mask

In [19]:
class GPTLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        
        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head) for _ in range(n_layer)]
        )
        
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, mask=None, targets=None):
        B, T = idx.shape
        
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(
            torch.arange(T, device=device)
        )
        
        x = tok_emb + pos_emb
        
        x, _ = self.blocks((x, mask))
        x = self.ln_f(x)
        
        logits = self.lm_head(x)
        
        if targets is None:
            return logits
        
        B, T, C = logits.shape
        
        logits = logits.view(B*T, C)
        targets = targets.view(B*T)
        
        loss = F.cross_entropy(
            logits,
            targets,
            ignore_index=PAD_IDX
        )
        
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=50):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1)
            idx = torch.cat((idx, next_token), dim=1)
        return idx

In [20]:
model = GPTLanguageModel(vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for x, y, mask in training_generator:
        x, y, mask = x.to(device), y.to(device), mask.to(device)
        
        logits, loss = model(x, mask, y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, loss = {total_loss/len(training_generator)}")


RuntimeError: [enforce fail at alloc_cpu.cpp:121] data. DefaultCPUAllocator: not enough memory: you tried to allocate 967680000 bytes.

In [ ]:
model.eval()

start = torch.tensor(
    [[tokenizer.token_to_id("[BOS]")]],
    device=device
)

generated = model.generate(start, max_new_tokens=30)[0].tolist()

print(tokenizer.decode(generated))